# 01a — DEM Prefetch & Pipeline Prep
## Cheyenne River corridor (Angostura → Oahe), South Dakota

**Run this once before `01_VBET_ValleyBottom.ipynb`.** Everything it produces is cached on
disk and skipped on re-run, so it is safe to re-execute at any time.

### Why this notebook exists

Notebook 01 previously called `py3dep.get_dem()` for the whole corridor in a single request.
`py3dep` talks to the USGS 3DEP **dynamic** map service, which renders elevation on the fly —
fine for a few hundred km², unworkable for the 6,225 km² corridor (a 228 × 187 km envelope).
That single call was the reported bottleneck.

Instead we pull the **static, pre-staged 3DEP COG tiles** straight from the USGS public S3
bucket. For this corridor that is eight 1° × 1° tiles at ~52 MB each — **~415 MB total** — which
downloads in a couple of minutes, resumes if interrupted, and never has to be fetched again.

### What this notebook produces

| Output | Purpose |
|---|---|
| `dem_tiles/USGS_1_*.tif` | Raw 1-arc-second 3DEP tiles (cached, resumable) |
| `cheyenne_dem_30m.tif` | Mosaicked, reprojected (UTM 13N), AOI-clipped DEM — **the** local DEM |
| `cheyenne_flowlines_vaa.gpkg` | Corridor flowlines with NHDPlus drainage area (`totdasqkm`) joined |
| `~/data-store/bin/WBT/` | Persistent WhiteboxTools binary (survives CyVerse session restarts) |

### Prerequisite
`00_Study_Area-Cottonwoods.ipynb` must have been run to produce
`data/cheyenne_corridor_aoi.gpkg`.

---
## 0. Setup

`PROJ_DATA` / `PROJ_LIB` are set before any geospatial import — the Jupyter kernel on CyVerse
starts without `conda activate`, so PROJ cannot find its database otherwise (see `CLAUDE.md`).

In [ ]:
import os, sys

# --- PROJ/GDAL data paths: must be set BEFORE any geospatial import ---
# The Jupyter kernel starts without `conda activate`, so these are otherwise unset.
# sys.prefix alone is not enough: in a venv layered on a conda env (the CyVerse
# HYR-SENSE overlay), sys.prefix is the venv, which has no share/proj — the data
# lives under sys.base_prefix. Check both and use whichever actually exists.
def _find_share(name):
    for base in (sys.prefix, sys.base_prefix):
        p = os.path.join(base, "share", name)
        if os.path.isdir(p):
            return p
    return None

_proj, _gdal = _find_share("proj"), _find_share("gdal")
if _proj:
    os.environ["PROJ_DATA"] = os.environ["PROJ_LIB"] = _proj
if _gdal:
    os.environ.setdefault("GDAL_DATA", _gdal)

import math
import shutil
import subprocess
import warnings
from pathlib import Path

import geopandas as gpd
import pandas as pd
from osgeo import gdal

gdal.UseExceptions()
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

print("Imports OK")
print(f"  GDAL   : {gdal.__version__}")
print(f"  PROJ_DATA: {os.environ['PROJ_DATA']}")

---
## 1. Configuration

`VBET_DATA_DIR` lets you redirect all I/O without editing the notebook. **On CyVerse set it to
a path under `~/data-store/`** — the container filesystem is ephemeral and everything outside
the data store is lost when the session ends:

```bash
export VBET_DATA_DIR=/home/jovyan/data-store/unci-maka-data
```

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
DATA_DIR = Path(os.environ.get("VBET_DATA_DIR", "../data")).expanduser()
DATA_DIR.mkdir(parents=True, exist_ok=True)

AOI_GPKG   = DATA_DIR / "cheyenne_corridor_aoi.gpkg"   # from notebook 00
TILE_DIR   = DATA_DIR / "dem_tiles"

# Buffer around the corridor AOI for the DEM. Must be >= the largest VBET class buffer
# (1000 m) plus room for the valley walls; 5 km is comfortable for the Cheyenne main stem.
BUFFER_KM  = 5

# 30 m == 3DEP 1-arc-second. (10 m == 1/3 arc-second; change PRODUCT below if you switch.)
DEM_RES_M  = 30
PRODUCT    = "1"          # "1" = 1 arc-sec (~30 m); "13" = 1/3 arc-sec (~10 m)

CRS_PROJ   = "EPSG:32613"  # UTM Zone 13N — covers the whole Cheyenne corridor

DEM_PATH        = DATA_DIR / f"cheyenne_dem_{DEM_RES_M}m.tif"
FLOWLINES_PATH  = DATA_DIR / "cheyenne_flowlines_vaa.gpkg"
WBT_PERSIST_DIR = Path.home() / "data-store" / "bin" / "WBT"

print(f"DATA_DIR : {DATA_DIR.resolve()}")
print(f"DEM out  : {DEM_PATH.name}  ({DEM_RES_M} m, {CRS_PROJ})")
if not AOI_GPKG.exists():
    raise FileNotFoundError(
        f"{AOI_GPKG} not found — run 00_Study_Area-Cottonwoods.ipynb first."
    )

---
## 2. AOI extent and 3DEP tile list

The tile list is **derived from the AOI**, not hardcoded, so this notebook still works if the
corridor definition in notebook 00 changes.

USGS staged elevation tiles are named by their **northwest corner** — `n44w104` covers
longitude −104 → −103 and latitude 43 → 44. So a point at (−103.5, 43.5) lives in tile
`n44w104`: latitude rounds *up*, longitude magnitude rounds *up*.

In [ ]:
# ---- Load AOI and build the buffered DEM footprint ----
aoi = gpd.read_file(AOI_GPKG, layer="study_area").to_crs(4326)
aoi_proj = aoi.to_crs(CRS_PROJ)

dem_aoi_proj = aoi_proj.copy()
dem_aoi_proj["geometry"] = aoi_proj.geometry.buffer(BUFFER_KM * 1000)
dem_aoi_wgs = dem_aoi_proj.to_crs(4326)

minx, miny, maxx, maxy = dem_aoi_wgs.total_bounds
print(f"AOI area           : {aoi_proj.area.sum() / 1e6:,.0f} km²")
print(f"Buffered ({BUFFER_KM} km) : {dem_aoi_proj.area.sum() / 1e6:,.0f} km²")
print(f"Buffered bbox WGS84: {minx:.4f}, {miny:.4f}, {maxx:.4f}, {maxy:.4f}")

pminx, pminy, pmaxx, pmaxy = dem_aoi_proj.total_bounds
print(f"Grid at {DEM_RES_M} m      : {int((pmaxx-pminx)/DEM_RES_M):,} x "
      f"{int((pmaxy-pminy)/DEM_RES_M):,} cells "
      f"({(pmaxx-pminx)/1000:.0f} x {(pmaxy-pminy)/1000:.0f} km)")

# Save the cutline for gdal.Warp
CUTLINE_PATH = DATA_DIR / "_dem_cutline.gpkg"
dem_aoi_wgs[["geometry"]].to_file(CUTLINE_PATH, layer="cutline", driver="GPKG")

In [ ]:
def tiles_for_bbox(minx, miny, maxx, maxy):
    '''1-degree 3DEP staged tile names covering a WGS84 bbox (CONUS / western hemisphere).

    Tiles are named by their NORTHWEST corner: n44w104 spans
    latitude [43, 44] and longitude [-104, -103].
    '''
    lat_hi = range(math.floor(miny) + 1, math.ceil(maxy) + 1)   # the "n" values
    lon_w  = range(math.floor(-maxx) + 1, math.ceil(-minx) + 1)  # the "w" values
    return [f"n{n:02d}w{w:03d}" for n in lat_hi for w in lon_w]

TILES = tiles_for_bbox(minx, miny, maxx, maxy)
BASE_URL = "https://prd-tnm.s3.amazonaws.com/StagedProducts/Elevation/{p}/TIFF/current/{t}/USGS_{p}_{t}.tif"

print(f"{len(TILES)} tile(s) needed for the buffered AOI:")
for t in TILES:
    print(f"  {t}  ->  {BASE_URL.format(p=PRODUCT, t=t)}")

---
## 3. Download the tiles (resumable)

`curl -C -` resumes a partial file rather than restarting it, and any tile already on disk with
a size matching the server's `Content-Length` is skipped entirely. Re-running this cell after an
interruption costs nothing.

In [ ]:
TILE_DIR.mkdir(parents=True, exist_ok=True)

def remote_size(url):
    '''Content-Length from a HEAD request, or None if unavailable.'''
    out = subprocess.run(["curl", "-sIL", url], capture_output=True, text=True).stdout
    for line in reversed(out.splitlines()):
        if line.lower().startswith("content-length:"):
            return int(line.split(":", 1)[1].strip())
    return None

tile_paths, missing = [], []
for t in TILES:
    url = BASE_URL.format(p=PRODUCT, t=t)
    dest = TILE_DIR / f"USGS_{PRODUCT}_{t}.tif"
    want = remote_size(url)

    if want is None:
        # A corner tile can legitimately not exist (ocean, outside CONUS coverage).
        # Warn and carry on — the mosaic just has nodata there, and the AOI cutline
        # usually clips it away anyway.
        print(f"  {t}: NOT AVAILABLE on 3DEP — skipping")
        missing.append(t)
        continue

    if dest.exists() and dest.stat().st_size == want:
        print(f"  {dest.name}: cached ({want / 1e6:.0f} MB)")
        tile_paths.append(str(dest))
        continue

    print(f"  {dest.name}: downloading ({want / 1e6:.0f} MB)…")
    r = subprocess.run(
        ["curl", "-fL", "-C", "-", "--retry", "3", "--retry-delay", "5",
         "-o", str(dest), url],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(f"Download failed for {t}: {r.stderr[-500:]}")
    print(f"    done ({dest.stat().st_size / 1e6:.0f} MB)")
    tile_paths.append(str(dest))

if not tile_paths:
    raise RuntimeError("No 3DEP tiles could be downloaded — check network / AOI bounds.")

total_mb = sum(Path(p).stat().st_size for p in tile_paths) / 1e6
print(f"\n{len(tile_paths)} tiles on disk, {total_mb:,.0f} MB total")
if missing:
    print(f"Unavailable tiles (nodata in mosaic): {', '.join(missing)}")

---
## 4. Mosaic → reproject → clip, in one GDAL pass

`BuildVRT` stitches the tiles virtually (no pixels copied), then a single `Warp` reprojects to
UTM 13N, resamples to 30 m, and clips to the buffered corridor. Doing it as one warp avoids
writing a full-size intermediate in geographic coordinates.

The output filename is exactly what notebook 01 already looks for, so no plumbing changes are
needed there.

In [ ]:
VRT_PATH = DATA_DIR / f"_dem_tiles_{PRODUCT}.vrt"

if DEM_PATH.exists():
    print(f"{DEM_PATH.name} already exists — delete it to rebuild.")
else:
    print("Building VRT mosaic…")
    gdal.BuildVRT(str(VRT_PATH), tile_paths)

    print(f"Warping to {CRS_PROJ} @ {DEM_RES_M} m and clipping to AOI…")
    gdal.Warp(
        str(DEM_PATH),
        str(VRT_PATH),
        dstSRS=CRS_PROJ,
        xRes=DEM_RES_M, yRes=DEM_RES_M,
        resampleAlg="bilinear",
        cutlineDSName=str(CUTLINE_PATH),
        cropToCutline=True,
        dstNodata=-9999.0,
        outputType=gdal.GDT_Float32,
        creationOptions=["COMPRESS=DEFLATE", "TILED=YES", "BIGTIFF=IF_SAFER",
                         "NUM_THREADS=ALL_CPUS"],
        multithread=True,
        warpMemoryLimit=512,
    )
    print(f"  wrote {DEM_PATH} ({DEM_PATH.stat().st_size / 1e6:.0f} MB)")

In [ ]:
# ---- Verify the mosaic ----
ds = gdal.Open(str(DEM_PATH))
band = ds.GetRasterBand(1)
stats = band.ComputeStatistics(0)   # min, max, mean, stddev
gt = ds.GetGeoTransform()

print(f"Size      : {ds.RasterXSize:,} x {ds.RasterYSize:,} "
      f"({ds.RasterXSize * ds.RasterYSize / 1e6:.0f} M cells)")
print(f"Pixel size: {gt[1]:.1f} x {abs(gt[5]):.1f} m")
print(f"CRS       : {ds.GetSpatialRef().GetAuthorityCode(None)}")
print(f"NoData    : {band.GetNoDataValue()}")
print(f"Elevation : {stats[0]:.1f} – {stats[1]:.1f} m (mean {stats[2]:.1f})")
ds = None

# Sanity check: the Cheyenne corridor runs roughly 500–1,500 m elevation
if not (300 < stats[0] < 1200 and 600 < stats[1] < 2200):
    print("\n  WARNING: elevation range looks wrong for this corridor — inspect the mosaic.")
else:
    print("\n  Elevation range is plausible for the Cheyenne corridor.")

---
## 5. Flowlines with drainage area

**This fixes a silent bug in notebook 01.** The `flowlines` layer written by notebook 00 carries
only `nhdplus_comid` — no drainage area. Notebook 01's VBET logic looks for `totdasqkm` and,
not finding it, falls through to a branch that assigns *every* reach to the `medium` class.

The practical effect: the Cheyenne main stem (drainage area in the thousands of km²) was being
delineated with headwater-scale thresholds — HAND 8 m and a 500 m search buffer instead of
12 m / 1000 m. The valley bottom on the main stem was systematically too narrow, and the
three-class table in notebook 01's documentation described behaviour that never ran.

Joining the NHDPlus Value Added Attributes on COMID restores the intended classing.

In [ ]:
if FLOWLINES_PATH.exists():
    flw = gpd.read_file(FLOWLINES_PATH, layer="flowlines")
    print(f"{FLOWLINES_PATH.name} already exists — {len(flw):,} reaches.")
else:
    from pynhd import nhdplus_vaa

    flw = gpd.read_file(AOI_GPKG, layer="flowlines")
    print(f"Corridor flowlines: {len(flw):,} reaches")
    print(f"  columns: {list(flw.columns)}")

    print("Fetching NHDPlus VAA table (cached parquet, ~1 GB first time)…")
    vaa = nhdplus_vaa()
    keep = [c for c in ["comid", "totdasqkm", "streamorde", "slope", "lengthkm"]
            if c in vaa.columns]
    vaa = vaa[keep]

    flw["nhdplus_comid"] = pd.to_numeric(flw["nhdplus_comid"], errors="coerce").astype("Int64")
    flw = flw.merge(vaa, left_on="nhdplus_comid", right_on="comid", how="left")

    flw.to_file(FLOWLINES_PATH, layer="flowlines", driver="GPKG")
    print(f"  wrote {FLOWLINES_PATH}")

In [ ]:
# ---- Verify the join and preview the resulting VBET classes ----
n = len(flw)
n_null = int(flw["totdasqkm"].isna().sum())
print(f"totdasqkm joined for {n - n_null:,} / {n:,} reaches "
      f"({100 * (n - n_null) / n:.1f}%)")
if n_null / n > 0.10:
    print("  WARNING: >10% of reaches have no drainage area. Notebook 01 will fall back to\n"
          "           streamorde bins for those reaches. Check the COMID join.")

print(f"\nDrainage area: {flw['totdasqkm'].min():.2f} – {flw['totdasqkm'].max():,.0f} km²")

bins = [(-1, 100, "small"), (100, 1000, "medium"), (1000, 1e9, "large")]
print("\nReaches per VBET class:")
for lo, hi, lab in bins:
    sel = flw[(flw["totdasqkm"] > lo) & (flw["totdasqkm"] <= hi)]
    km = sel.to_crs(CRS_PROJ).geometry.length.sum() / 1000 if len(sel) else 0
    print(f"  {lab:6s}: {len(sel):6,} reaches, {km:8,.0f} km")
print(f"  {'null':6s}: {n_null:6,} reaches")

---
## 6. WhiteboxTools — put the binary somewhere persistent

Notebook 01 uses [WhiteboxTools](https://www.whiteboxgeo.com/) for the hydrology
(depression breaching, flow direction, flow accumulation, HAND, slope). It is a multithreaded
Rust engine that streams rasters from disk, so it does not hold several full-grid float32 arrays
in Python memory the way `pysheds` does.

**CyVerse gotcha:** `WhiteboxTools()` calls `download_wbt()` from its constructor, which fetches
a ~200 MB binary into the `whitebox` package directory under `/opt/conda`. That directory does
*not* survive a session restart, so by default you re-download it every single session.

Two details from the package source make this avoidable:

- `download_wbt()` returns immediately if the **`WBT_PATH`** environment variable is set. That is
  the supported way to say "it's already installed, don't fetch".
- `set_whitebox_dir(d)` sets `exe_path = d`, where `exe_path` is the **directory** containing the
  `whitebox_tools` executable along with its `plugins/` and `img/` subdirectories — not a path to
  the executable file itself.

So: copy that directory into `~/data-store/bin/WBT` once, and from then on set `WBT_PATH` before
constructing `WhiteboxTools`.

In [ ]:
WBT_EXE = "whitebox_tools.exe" if sys.platform.startswith("win") else "whitebox_tools"
have_persisted = (WBT_PERSIST_DIR / WBT_EXE).exists()

# Must be set BEFORE constructing WhiteboxTools — the constructor calls download_wbt(),
# which short-circuits on this variable.
if have_persisted:
    os.environ["WBT_PATH"] = str(WBT_PERSIST_DIR)

try:
    import whitebox
except ImportError:
    raise ImportError(
        "whitebox is not installed.\n"
        "  In a CyVerse session:  mamba install -c conda-forge whitebox\n"
        "  (it is already listed in the root environment.yml)"
    )

if have_persisted:
    print(f"Reusing persistent WhiteboxTools at {WBT_PERSIST_DIR} (no download)")
else:
    print("No persisted binary found — WhiteboxTools will download ~200 MB on first use.")

wbt = whitebox.WhiteboxTools()
wbt.verbose = False
if have_persisted:
    wbt.set_whitebox_dir(str(WBT_PERSIST_DIR))
wbt.set_max_procs(-1)   # -1 = all available cores

print(f"\n{wbt.version().splitlines()[0].strip()}")
print(f"  exe_path (directory): {wbt.exe_path}")

In [ ]:
# ---- One-time: copy the binary + plugins into the persistent data store ----
# Safe to re-run; does nothing once the copy exists.
#
# exe_path is the DIRECTORY holding whitebox_tools, plugins/ and img/ — copy the whole
# thing, not just the executable, or the tools that ship as plugins will fail to launch.
src_dir = Path(wbt.exe_path)

if have_persisted:
    print(f"Already persisted at {WBT_PERSIST_DIR}")
elif not WBT_PERSIST_DIR.parent.parent.exists():
    print(f"No data store at {WBT_PERSIST_DIR.parent.parent} — skipping.")
    print("(This step only matters on CyVerse, where /opt/conda is wiped each session.)")
elif not (src_dir / WBT_EXE).exists():
    print(f"Could not find {WBT_EXE} in {src_dir} — skipping the copy.")
    print("Run a WhiteboxTools tool once to trigger the download, then re-run this cell.")
else:
    WBT_PERSIST_DIR.parent.mkdir(parents=True, exist_ok=True)
    print(f"Copying {src_dir} -> {WBT_PERSIST_DIR} …")
    shutil.copytree(src_dir, WBT_PERSIST_DIR, dirs_exist_ok=True)

    # copytree does not preserve the exec bit reliably across filesystems
    for f in [WBT_PERSIST_DIR / WBT_EXE, *(WBT_PERSIST_DIR / "plugins").glob("*")]:
        if f.is_file() and f.suffix != ".json":
            f.chmod(0o755)

    os.environ["WBT_PATH"] = str(WBT_PERSIST_DIR)
    wbt.set_whitebox_dir(str(WBT_PERSIST_DIR))
    size_mb = sum(f.stat().st_size for f in WBT_PERSIST_DIR.rglob("*") if f.is_file()) / 1e6
    print(f"Done ({size_mb:,.0f} MB). Future sessions skip the download.")

print("\nIn later sessions, notebook 01 checks this path automatically. To make it apply")
print("everywhere, add to your CyVerse shell profile:")
print(f'    export WBT_PATH={WBT_PERSIST_DIR}')

---
## Done — prep summary

Run this cell to confirm everything notebook 01 needs is in place.

In [ ]:
checks = [
    ("DEM mosaic",        DEM_PATH),
    ("Flowlines + VAA",   FLOWLINES_PATH),
    ("Corridor AOI",      AOI_GPKG),
]
print("=" * 62)
print("PREFETCH SUMMARY")
print("=" * 62)
for label, p in checks:
    mark = "OK  " if p.exists() else "MISS"
    size = f"{p.stat().st_size / 1e6:8,.1f} MB" if p.exists() else " " * 11
    print(f"  [{mark}] {label:18s} {size}  {p.name}")
print(f"  [{'OK  ' if TILE_DIR.exists() else 'MISS'}] {'Raw 3DEP tiles':18s} "
      f"{sum(f.stat().st_size for f in TILE_DIR.glob('*.tif')) / 1e6:8,.1f} MB  "
      f"{len(list(TILE_DIR.glob('*.tif')))} tiles")
print("=" * 62)
print("\nNext: run 01_VBET_ValleyBottom.ipynb")
if "VBET_DATA_DIR" not in os.environ:
    print("\nNote: VBET_DATA_DIR is unset, so paths resolve to ../data.")
    print("      On CyVerse, export it to a ~/data-store/ path in BOTH notebooks.")